In [48]:
req.libs <- c("Seurat", "zellkonverter", "SingleCellExperiment", "data.table", "future.apply")
lapply(req.libs, function(x){
    suppressMessages(library(x, character.only = TRUE))})

[[1]]
 [1] "data.table"           "SingleCellExperiment" "SummarizedExperiment"
 [4] "Biobase"              "GenomicRanges"        "GenomeInfoDb"        
 [7] "IRanges"              "S4Vectors"            "BiocGenerics"        
[10] "stats4"               "MatrixGenerics"       "matrixStats"         
[13] "zellkonverter"        "Seurat"               "SeuratObject"        
[16] "sp"                   "stats"                "graphics"            
[19] "grDevices"            "utils"                "datasets"            
[22] "methods"              "base"                

[[2]]
 [1] "data.table"           "SingleCellExperiment" "SummarizedExperiment"
 [4] "Biobase"              "GenomicRanges"        "GenomeInfoDb"        
 [7] "IRanges"              "S4Vectors"            "BiocGenerics"        
[10] "stats4"               "MatrixGenerics"       "matrixStats"         
[13] "zellkonverter"        "Seurat"               "SeuratObject"        
[16] "sp"                   "stats"                "graphics"            
[19] "grDevices"            "utils"                "datasets"            
[22] "methods"              "base"                

[[3]]
 [1] "data.table"           "SingleCellExperiment" "SummarizedExperiment"
 [4] "Biobase"              "GenomicRanges"        "GenomeInfoDb"        
 [7] "IRanges"              "S4Vectors"            "BiocGenerics"        
[10] "stats4"               "MatrixGenerics"       "matrixStats"         
[13] "zellkonverter"        "Seurat"               "SeuratObject"        
[16] "sp"                   "stats"                "graphics"            
[19] "grDevices"            "utils"                "datasets"            
[22] "methods"              "base"                

[[4]]
 [1] "data.table"           "SingleCellExperiment" "SummarizedExperiment"
 [4] "Biobase"              "GenomicRanges"        "GenomeInfoDb"        
 [7] "IRanges"              "S4Vectors"            "BiocGenerics"        
[10] "stats4"               "MatrixGenerics"       "matrixStats"         
[13] "zellkonverter"        "Seurat"               "SeuratObject"        
[16] "sp"                   "stats"                "graphics"            
[19] "grDevices"            "utils"                "datasets"            
[22] "methods"              "base"                

[[5]]
 [1] "future.apply"         "future"               "data.table"          
 [4] "SingleCellExperiment" "SummarizedExperiment" "Biobase"             
 [7] "GenomicRanges"        "GenomeInfoDb"         "IRanges"             
[10] "S4Vectors"            "BiocGenerics"         "stats4"              
[13] "MatrixGenerics"       "matrixStats"          "zellkonverter"       
[16] "Seurat"               "SeuratObject"         "sp"                  
[19] "stats"                "graphics"             "grDevices"           
[22] "utils"                "datasets"             "methods"             
[25] "base"

In [49]:
meta <- fread("/mnt/users/rachelsavage/paper_minimorf/data/initial_filtering/atac.filtered.metadata.tsv")
gene.info <- fread("~/paper_minimorf//data/reference_data//ensg.name.txt", header = FALSE)

In [50]:
rna.dir <- "/fab/AlignedData/rsav_minimorf/igvf_pipeline/"
barcode_files <- list.files(
  path = rna.dir,
  pattern = ".h5ad$",  # regex for exact match
  recursive = TRUE,                         # search subdirectories
  full.names = TRUE                         # return full paths
)

In [57]:
results <- lapply(barcode_files, function(x){
  tryCatch({
    rna <- readH5AD(x)
    rna <- SingleCellExperiment(assays = list(X = assay(rna, "X")))
    rownames(rna) <- gene.info$V2
    rna <- rna[, colnames(rna) %in% meta$barcode]
    return(rna)
  }, error = function(e) {
    message("Error reading file: ", x)
    return(NULL)
  })
})

In [60]:
combined_sce <- do.call(cbind, results)

In [61]:
dim(combined_sce)

[1]  62757 762374

In [66]:
meta.rna <- as.data.frame(Matrix::colSums(combined_sce@assays@data$X))

In [68]:
colnames(meta.rna) <- "unique.mapped.rna.reads"
meta.rna$barcode <- rownames(meta.rna)
rownames(meta.rna) <- NULL

In [72]:
fwrite(meta.rna, "/mnt/users/rachelsavage/paper_minimorf/data/initial_filtering/rna.filtered.metadata.tsv")
saveRDS(combined_sce, "/mnt/users/rachelsavage/paper_minimorf/data/initial_filtering/rna.sce.rds")
writeH5AD(combined_sce, "/mnt/users/rachelsavage/paper_minimorf/data/rna_matrices/allrna.h5ad")

In [ ]:
asdf <- readRDS("/mnt/users/rachelsavage/paper_minimorf/data/initial_filtering/rna.sce.rds")
cellbcs <- colnames(asdf)
sublibs <- stri_split_fixed(cellbcs, pattern = "_", simplify = TRUE)[,2]
bc.df <- data.frame(barcode = cellbcs, sublibrary = sublibs)
pbmclapply(unique(bc.df$sublibrary), function(sublib){
bc <- bc.df %>% filter(sublibrary == sublib) %>% pull(barcode)
subset <- asdf[,bc]
writeH5AD(subset, paste0("/mnt/users/rachelsavage/paper_minimorf/data/rna_matrices/h5ad_sublibrary_split/", sublib, ".rna.h5ad")
)
}, mc.cores = 8)